# 🦀 Day 4: Async Traits, Tower Service Abstraction
---

Today we explore async traits (now stable with RPITIT) and the Tower service abstraction.

- **Async traits in Rust**: `async fn` in traits — now stable!
- **Trait Service**: `call(&self, req: Request) -> Future<Response>`
- **Tower**: Composable middleware — layers wrap services
- **Middleware layers**: Timeout, rate limiting, retry, logging
- **ServiceBuilder**: Composing layers
- **Axum/Tonic**: Use Tower under the hood

## Tower Service Trait

```rust
pub trait Service<Request> {
    type Response;
    type Error;
    type Future: Future<Output = Result<Self::Response, Self::Error>>;
    fn call(&self, req: Request) -> Self::Future;
}
```

A **Layer** wraps a service and returns a new service. Layers compose: `TimeoutLayer` → `LogLayer` → `YourService`.

In [ ]:
// Simplified Service trait (synchronous for EvCxR demo)
// Real Tower uses async and Future

trait SimpleService<Req, Res> {
    fn call(&self, req: Req) -> Res;
}

struct EchoService;

impl SimpleService<String, String> for EchoService {
    fn call(&self, req: String) -> String {
        format!("Echo: {}", req)
    }
}

// A "layer" that transforms the service
struct UppercaseLayer<S>(S);

impl<S, Req, Res> SimpleService<Req, String> for UppercaseLayer<S>
where
    S: SimpleService<Req, Res>,
    Res: std::fmt::Display,
{
    fn call(&self, req: Req) -> String {
        self.0.call(req).to_string().to_uppercase()
    }
}

let echo = EchoService;
let wrapped = UppercaseLayer(echo);
println!("{}", wrapped.call("hello".to_string()));

## ServiceBuilder Pattern

```rust
ServiceBuilder::new()
    .layer(TimeoutLayer::new(Duration::from_secs(5)))
    .layer(LogLayer)
    .service(my_service)
```

Layers are applied bottom-to-top: request flows through timeout → log → service.

In [ ]:
// YOUR CODE HERE
// Build a custom Tower-style layer.
// Example: a "prefix" layer that adds a prefix to the response.

struct PrefixLayer<S>(String, S);

impl<S, Req, Res> SimpleService<Req, String> for PrefixLayer<S>
where
    S: SimpleService<Req, Res>,
    Res: std::fmt::Display,
{
    fn call(&self, req: Req) -> String {
        // YOUR CODE HERE: add self.0 as prefix to the response
        format!("{} {}", self.0, self.1.call(req))
    }
}

let echo = EchoService;
let prefixed = PrefixLayer("[LOG]".to_string(), echo);
println!("{}", prefixed.call("test".to_string()));

## 🎯 Key Takeaways

1. Async traits are now stable (RPITIT) — `async fn` in traits works
2. Tower's `Service` trait: `call(&self, req) -> Future<Response>`
3. Layers wrap services — timeout, retry, logging, rate limiting
4. ServiceBuilder composes layers in a clean, declarative way
5. Axum and Tonic use Tower for middleware
6. Build custom layers to add cross-cutting concerns

---
**Tomorrow:** Repository review & organization